# Warm server local lifecycle smoke test

Run this notebook twice. The first pass starts the lightweight stand-in server; the second must reuse it.

In [ ]:
import json
import os
import sys
from pathlib import Path

project = Path.cwd()
if not (project / "mvp_v2").is_dir():
    project = project.parent
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from mvp_v2.harness import VLLMServer

root = Path(os.environ.get("WARM_SMOKE_ROOT", "/tmp/gemma-warm-server-smoke"))
port = int(os.environ.get("WARM_SMOKE_PORT", "18765"))
server = VLLMServer(
    model_path=Path("/fake-model"),
    model_name="fake-model",
    log_path=root / "server.log",
    port=port,
    startup_timeout=15,
)
server.command = [
    sys.executable, str(project / "tests" / "fake_vllm_server.py"),
    str(port), str(root), "fake-model",
]
profile = server.ensure_started()
result = {
    "reused": profile["reused"],
    "startup_seconds": profile["startup_seconds"],
    "starts": int((root / "starts.txt").read_text()),
}
(root / "last_profile.json").write_text(json.dumps(result, indent=2) + "\n")
print(json.dumps(result, sort_keys=True))
